<a href="https://colab.research.google.com/github/mohamadfaisalbashir/grokking-Deep-Learning/blob/main/04_introduction_to_neural_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Chapter 4: Introduction to Neural Learning - Gradient Descent**

This notebook summarizes Chapter 4 of *Grokking Deep Learning*:

1. Predict, Compare, and Learn
2. Compare: Does Your Network Make Good Predictions?
3. Why Measure Error?
4. The Simplest Form of Neural Learning: Hot and Cold
5. Hot and Cold Learning in Code
6. Calculating Both Direction and Amount from Error
7. One Iteration of Gradient Descent
8. Learning Is Just Reducing Error
9. Watching Several Steps of Learning
10. Why Does This Work? What Is `weight_delta`, Really?
11. Derivatives and How to Use Them to Learn
12. Breaking Gradient Descent
13. Divergence and Alpha
14. Memorizing
15. Summary

Chapter 3 covered the first step of the *predict, compare, learn* paradigm. This chapter addresses the question that naturally follows: **how are weight values set so that the network predicts accurately?** It covers the remaining two steps, *compare* and *learn*, and culminates in **gradient descent**, the most popular learning method in deep learning.

The chapter opens with a remark attributed to Milton Friedman (*Essays in Positive Economics*, 1953): the only relevant test of a hypothesis is to compare its predictions with experience.

---
## **1. Predict, Compare, and Learn**

In Chapter 3, the first step, *predict*, was examined in depth. Along the way the major parts of a neural network (nodes and weights) were introduced, together with how datasets fit into networks (matching the number of datapoints arriving at one time) and how a network makes a prediction.

The natural follow-up question is how to set the weight values so that the network predicts accurately. Answering it is the main focus of this chapter, which covers the next two steps of the paradigm: **compare** and **learn**.

### **1.1 Compare**

**Comparing gives a measurement of how much a prediction "missed" by.**

Once a prediction has been made, the next step is to evaluate how well the network did. This sounds simple, but measuring error well is one of the most important and most complicated subjects in deep learning.

Several properties of error measurement are things people do every day without noticing: for example, amplifying bigger errors while ignoring very small ones. This chapter shows how to teach a network to do the same mathematically. It also shows that **error is always positive**. The analogy is an archer hitting a target: whether the shot lands an inch too low or an inch too high, the error is still just one inch.

Only one simple way of measuring error is used in this chapter: **mean squared error**. It is one of many ways to evaluate the accuracy of a network.

The output of the *compare* logic is a **"hot or cold"** signal. Given a prediction, an error measure says either "a lot" or "a little". It does not say *why* the prediction missed, or in which direction, or what to do about it. It roughly says "big miss", "little miss", or "perfect prediction". What to do about the error is the job of the next step.

### **1.2 Learn**

**Learning tells each weight how it can change to reduce the error.**

Learning is all about **error attribution**: figuring out how each weight played its part in creating the error. The author calls it "the blame game of deep learning", and this chapter spends many pages on its most popular version, **gradient descent**.

In the end, learning computes **one number for each weight**. That number says how the weight should be higher or lower in order to reduce the error. The weight is then moved according to that number, and the step is complete.

---
## **2. Compare: Does Your Network Make Good Predictions?**

**Let's measure the error and find out!**

The following code measures the error of a single prediction. It should print `0.3025`.

> Measuring the error of one prediction
>
> <img src="pictures/chapter 4/picture 4.1.png" alt="Figure 4.1" width="600"/>


In [1]:
knob_weight = 0.5
input = 0.5
goal_pred = 0.8

pred = input * knob_weight

error = (pred - goal_pred) ** 2

print(error)

0.30250000000000005


Three common questions about this code:

- **What is the `goal_pred` variable?** Like `input`, it is a number recorded somewhere in the real world, but it is usually something hard to observe, such as "the percentage of people who *did* wear sweatsuits" given the temperature, or "whether the batter *did* hit a home run" given his batting average.
- **Why is the error squared?** Think of an archer: a shot 2 inches too high and a shot 2 inches too low both miss by 2 inches. The main reason to square "how much you missed" is to force the output to be **positive**. The quantity `(pred - goal_pred)` could be negative in some situations, unlike actual error.
- **Doesn't squaring make big errors (> 1) bigger and small errors (< 1) smaller?** Yes, and that is acceptable. Amplifying big errors and reducing small ones helps the network *pay attention* to the big errors and not worry much about small ones. Good parents are similar: they practically ignore small errors (breaking the lead on a pencil) but react strongly to big ones (crashing the car).

---
## **3. Why Measure Error?**

**Measuring error simplifies the problem.**

The goal of training is to make correct predictions. In the most pragmatic setting, a network takes input that is easy to calculate (today's stock price) and predicts something hard to calculate (tomorrow's stock price). That is what makes a network useful.

Changing `knob_weight` so that the network predicts `goal_prediction` is slightly more complicated than changing `knob_weight` so that `error == 0`. Both statements say the same thing, but "get the error to 0" is more straightforward.

### **3.1 Different Ways of Measuring Error Prioritize Error Differently**

By squaring the error, numbers smaller than 1 get *smaller*, while numbers greater than 1 get *bigger*. The *pure error* (`pred - goal_pred`) is changed so that bigger errors become very big and smaller errors quickly become irrelevant.

- A fairly large pure error such as 10 becomes 10**2 = 100, a very large error.
- A small pure error such as 0.01 becomes 0.01**2 = 0.0001, a very small error.

This **prioritizes** big errors over small ones. If the *absolute value* were used instead of squaring, there would be no such prioritization: the error would simply be the positive version of the pure error, which is acceptable but different. More on this later.

### **3.2 Why Do You Want Only Positive Error?**

Eventually a network works with millions of `input -> goal_prediction` pairs and tries to bring the **average error** down to 0. This is a problem if errors can be both positive and negative. Suppose two datapoints produce errors of +1,000 and -1,000: the average error would be zero, creating the false impression of a perfect prediction when each prediction missed by 1,000. Therefore the error of *each prediction* must always be positive, so that errors do not cancel out when averaged.

---
## **4. The Simplest Form of Neural Learning: Hot and Cold**

**Learning using the hot and cold method.**

At the end of the day, learning is really about one thing: adjusting `knob_weight` up or down so that the error is reduced. If this is repeated until the error reaches 0, learning is complete. To know whether to turn the knob up or down, try **both** and see which reduces the error. Whichever does is used to update `knob_weight`.

> **Hot and cold learning** means wiggling the weights to see which direction reduces the error the most, moving the weights in that direction, and repeating until the error gets to 0.

One iteration of hot and cold learning has five stages. The code below runs them one by one.

> Hot and cold learning: empty network and predict (stages 1 and 2)
>
> <img src="pictures/chapter 4/picture 4.2.png" alt="Figure 4.2" width="600"/>

**Stages 1 and 2: an empty network, then predict and evaluate the error.**

In [2]:
weight = 0.1
lr = 0.01

def neural_network(input, weight):
    prediction = input * weight
    return prediction

In [3]:
number_of_toes = [8.5]
win_or_lose_binary = [1] # (won!!!)

input = number_of_toes[0]
true = win_or_lose_binary[0]

pred = neural_network(input,weight)

error = (pred - true) ** 2
print(error)

0.022499999999999975


**Stages 3 and 4: predict again with a higher weight and a lower weight.** The weight is moved up and down using `weight+lr` and `weight-lr`, and the error is evaluated for each.

> Hot and cold learning: compare and learn (stages 3, 4, and 5)
>
> <img src="pictures/chapter 4/picture 4.3.png" alt="Figure 4.3" width="600"/>


In [4]:
p_up = neural_network(input,weight+lr)
e_up = (p_up - true) ** 2
print(e_up)

p_dn = neural_network(input,weight-lr)
e_dn = (p_dn - true) ** 2
print(e_dn)

0.004224999999999993
0.05522499999999994


*Note:* the book's figure for stage 3 shows `lr = 0.1`, but the displayed weight (.11) and error (.004) only match `lr = 0.01`, which is the value used in stage 1 and stage 4. The code here uses `lr = 0.01` throughout.

**Stage 5: compare the errors and set the new weight.** The weight moves in whichever direction gives the lower error.

In [5]:
if(error > e_dn or error > e_up):
    if(e_dn < e_up):
        weight -= lr
    if(e_up < e_dn):
        weight += lr

*Note:* the book's printed listing for this stage contains typos (`||` and `e_up < e_up`); the version above is the corrected Python equivalent.

These five stages are **one iteration** of hot and cold learning. Here the iteration already gets very close to the correct answer (the new error is only 0.004), but normally the process must be repeated many times, and some people train their networks for weeks or months before finding a good enough weight configuration.

This reveals what learning in neural networks really is: a **search problem**. The network searches for the best possible configuration of weights so that its error falls to 0 and it predicts perfectly. As with any search, the exact target may not be found, and even if it is, it may take some time.

---
## **5. Hot and Cold Learning in Code**

**This is perhaps the simplest form of learning.**

The code below attempts to correctly predict **0.8**. The new parts relative to the network of earlier chapters are `step_amount` (how much the weight moves each iteration) and the `for` loop (repeating the learning many times so the error keeps getting smaller). Running it prints 1,101 lines, ending near `Error:1.07995057925e-27 Prediction:0.8`.

In [6]:
weight = 0.5
input = 0.5
goal_prediction = 0.8

step_amount = 0.001

for iteration in range(1101):

    prediction = input * weight
    error = (prediction - goal_prediction) ** 2

    print("Error:" + str(error) + " Prediction:" + str(prediction))

    up_prediction = input * (weight + step_amount)
    up_error = (goal_prediction - up_prediction) ** 2

    down_prediction = input * (weight - step_amount)
    down_error = (goal_prediction - down_prediction) ** 2

    if(down_error < up_error):
        weight = weight - step_amount

    if(down_error > up_error):
        weight = weight + step_amount

Error:0.30250000000000005 Prediction:0.25
Error:0.3019502500000001 Prediction:0.2505
Error:0.30140100000000003 Prediction:0.251
Error:0.30085225 Prediction:0.2515
Error:0.30030400000000007 Prediction:0.252
Error:0.2997562500000001 Prediction:0.2525
Error:0.29920900000000006 Prediction:0.253
Error:0.29866224999999996 Prediction:0.2535
Error:0.29811600000000005 Prediction:0.254
Error:0.2975702500000001 Prediction:0.2545
Error:0.29702500000000004 Prediction:0.255
Error:0.29648025 Prediction:0.2555
Error:0.29593600000000003 Prediction:0.256
Error:0.2953922500000001 Prediction:0.2565
Error:0.294849 Prediction:0.257
Error:0.29430625 Prediction:0.2575
Error:0.293764 Prediction:0.258
Error:0.2932222500000001 Prediction:0.2585
Error:0.292681 Prediction:0.259
Error:0.29214025 Prediction:0.2595
Error:0.2916 Prediction:0.26
Error:0.2910602500000001 Prediction:0.2605
Error:0.29052100000000003 Prediction:0.261
Error:0.28998225 Prediction:0.2615
Error:0.28944400000000003 Prediction:0.262
Error:0.2889

### **5.1 Characteristics of Hot and Cold Learning**

**It is simple.** After a prediction, the network predicts two more times, once with a slightly higher weight and once with a slightly lower one, and then moves `weight` in the direction that gave the smaller `error`. Repeating this enough times eventually reduces `error` to 0.

*Why exactly 1,101 iterations?* The network reaches 0.8 after about that many. Beyond that, it wiggles back and forth around 0.8, which gives a less tidy error log.

**Problem 1: it is inefficient.** Several predictions are needed to make a single `weight` update. The network predicts three times for each update.

**Problem 2: sometimes it is impossible to predict the exact goal.** With a fixed `step_amount`, unless the perfect `weight` is exactly `n * step_amount` away, the network will overshoot by some amount and then alternate back and forth on either side of `goal_prediction`. Setting `step_amount` to 0.2 shows this alternation. Setting it to 10 breaks the network completely: the error alternates between 0.3025 and 19.8025 and never approaches 0.8.

The real problem is that, although the correct *direction* to move `weight` is known, the correct *amount* is not. A fixed amount is picked at random (`step_amount`), and it has **nothing to do with `error`**: it is the same whether the error is big or tiny. The question that motivates the next section is: *what if both the direction and the amount could be computed for each weight without repeatedly making predictions?*

---
## **6. Calculating Both Direction and Amount from Error**

**Let's measure the error and find the direction and amount!**

The code below is a superior form of learning known as **gradient descent**. In a single line (`direction_and_amount`), it calculates both the *direction* and the *amount* by which `weight` should change to reduce `error`.

> Gradient descent: calculating direction_and_amount
>
> <img src="pictures/chapter 4/picture 4.4.png" alt="Figure 4.4" width="600"/>


In [7]:
weight = 0.5
goal_pred = 0.8
input = 0.5

for iteration in range(20):
    pred = input * weight
    error = (pred - goal_pred) ** 2
    direction_and_amount = (pred - goal_pred) * input
    weight = weight - direction_and_amount

    print("Error:" + str(error) + " Prediction:" + str(pred))

Error:0.30250000000000005 Prediction:0.25
Error:0.17015625000000004 Prediction:0.3875
Error:0.095712890625 Prediction:0.49062500000000003
Error:0.05383850097656251 Prediction:0.56796875
Error:0.03028415679931642 Prediction:0.6259765625
Error:0.0170348381996155 Prediction:0.669482421875
Error:0.00958209648728372 Prediction:0.70211181640625
Error:0.005389929274097089 Prediction:0.7265838623046875
Error:0.0030318352166796153 Prediction:0.7449378967285156
Error:0.0017054073093822882 Prediction:0.7587034225463867
Error:0.0009592916115275371 Prediction:0.76902756690979
Error:0.0005396015314842384 Prediction:0.7767706751823426
Error:0.000303525861459885 Prediction:0.7825780063867569
Error:0.00017073329707118678 Prediction:0.7869335047900676
Error:9.603747960254256e-05 Prediction:0.7902001285925507
Error:5.402108227642978e-05 Prediction:0.7926500964444131
Error:3.038685878049206e-05 Prediction:0.7944875723333098
Error:1.7092608064027242e-05 Prediction:0.7958656792499823
Error:9.614592036015323

The output begins with `Error:0.3025 Prediction:0.25` and ends close to `Prediction:0.7977`, so the final steps correctly approach 0.8.

### **6.1 What Is `direction_and_amount`?**

`direction_and_amount` represents how `weight` should change. It has two parts:

1. The **pure error**, `(pred - goal_pred)`. It indicates the raw direction and amount of the miss. A *positive* value means the prediction was too *high* (and vice versa); a *big* value means a big miss.
2. The multiplication by **`input`**, which performs **scaling, negative reversal, and stopping**. Together these translate the pure error into the absolute amount `weight` should change. They handle three edge cases where the pure error alone is not enough.

**Stopping.** This is the simplest effect. Imagine plugging a CD player into a stereo: if the volume is turned all the way up but the CD player is off, the volume change does not matter. If `input` is 0, `direction_and_amount` is forced to 0 as well. Nothing is learned (no change to the volume) because there is nothing to learn: every `weight` value gives the same `error`, since `pred` is always 0.

**Negative reversal.** This is probably the most difficult and most important effect. Normally (with positive `input`), moving `weight` up moves the prediction up. If `input` is *negative*, the direction reverses: moving `weight` up makes the prediction go *down*. Multiplying the pure error by `input` reverses the sign of `direction_and_amount` when `input` is negative, so that `weight` always moves in the correct direction.

**Scaling.** If `input` is big, the `weight` update should also be big. This is more of a side effect, since it often goes out of control. Later, `alpha` is used to address that.

---
## **7. One Iteration of Gradient Descent**

**This performs a weight update on a single training example (input -> true) pair.**

The same learning is now written in a form that more obviously applies to other kinds of networks (such as those with multiple inputs and outputs), with slightly different terminology. The primary difference is the new variable **`delta`**: the raw amount by which the output node was too high or too low. Instead of computing `direction_and_amount` directly, `delta` is computed first, and only then is the weight change derived from it (now called `weight_delta`).

The five stages are:

1. **An empty network.** `weight` and `alpha` are defined.
2. **Predict.** Make a prediction and evaluate the error (here 0.023).
3. **Compare.** Calculate the node delta, `delta = pred - goal_pred`. The true value is 1.0 and the prediction was 0.85, so the network was too *low* by 0.15 and `delta` is *negative* 0.15.
4. **Learn.** Calculate the weight delta, `weight_delta = input * delta`. This measures how much the weight caused the network to miss. It is created by *scaling* the output node's `delta` by the weight's `input`, which accounts for the scaling, negative reversal, and stopping described above.
5. **Learn.** Update the weight, `weight -= weight_delta * alpha`. Multiplying by the small number `alpha` (fixed before training) controls how fast the network learns; if it learns too fast, it can update weights too aggressively and overshoot. The resulting weight change is the same small increase made by hot and cold learning.

> One iteration of gradient descent: empty network, predict, and node delta (stages 1 to 3)
>
> <img src="pictures/chapter 4/picture 4.5.png" alt="Figure 4.5" width="600"/>

> One iteration of gradient descent: weight delta and weight update (stages 4 and 5)
>
> <img src="pictures/chapter 4/picture 4.6.png" alt="Figure 4.6" width="600"/>


In [8]:
weight = 0.1
alpha = 0.01

def neural_network(input, weight):
    prediction = input * weight
    return prediction

number_of_toes = [8.5]
win_or_lose_binary = [1] # (won!!!)

input = number_of_toes[0]
goal_pred = win_or_lose_binary[0]

pred = neural_network(input,weight)
error = (pred - goal_pred) ** 2

delta = pred - goal_pred

weight_delta = input * delta

weight -= weight_delta * alpha

---
## **8. Learning Is Just Reducing Error**

**You can modify `weight` to reduce `error`.**

Putting together the code from the previous pages gives the following loop. The lines for `pred` and `error` "have a secret".

In [9]:
weight, goal_pred, input = (0.0, 0.8, 0.5)

for iteration in range(4):

    pred = input * weight
    error = (pred - goal_pred) ** 2
    delta = pred - goal_pred
    weight_delta = delta * input
    weight = weight - weight_delta
    print("Error:" + str(error) + " Prediction:" + str(pred))

Error:0.6400000000000001 Prediction:0.0
Error:0.3600000000000001 Prediction:0.2
Error:0.2025 Prediction:0.35000000000000003
Error:0.11390625000000001 Prediction:0.4625


> **The golden method for learning:** adjust each `weight` in the correct direction and by the correct amount so that `error` reduces to 0.

All that is needed is the right direction and amount to modify `weight` so that `error` goes down. The secret lies in the `pred` and `error` calculations: `pred` is used *inside* the `error` calculation. Replacing the `pred` variable with the code that generates it does not change the value of `error`:

```python
error = ((input * weight) - goal_pred) ** 2
```

Since `input` and `goal_pred` are fixed at 0.5 and 0.8 before training starts, substituting their values makes the secret clear:

```python
error = ((0.5 * weight) - 0.8) ** 2
```

> **The secret:** for any `input` and `goal_pred`, an *exact relationship* is defined between `error` and `weight`, found by combining the prediction and error formulas.

If `weight` is increased by 0.5, this exact relationship makes it possible to calculate how much `error` moves as well. And if `error` should be moved in a specific direction, that can be done too.

Plotting the `error` for every `weight` produces a nice **bowl shape**. The black dot marks the current `weight` and `error`, and the dotted circle at the bottom is where the network wants to be (`error == 0`).

> The error curve (the bowl) with slope
>
> <img src="pictures/chapter 4/picture 4.7.png" alt="Figure 4.7" width="600"/>

> **Key takeaway:** the slope points to the *bottom* of the bowl (lowest `error`) no matter where you are in the bowl. This slope can be used to help the network reduce the error.

---
## **9. Watching Several Steps of Learning**

**Will we eventually find the bottom of the bowl?**

The following code runs four steps with `input = 1.1`, and prints the weight, error, prediction, delta, and weight delta at each step. Follow the weight moving along the bowl:

1. **A big weight increase.** At `weight = 0.0` the error is 0.64, so the weight is increased by a large step.
2. **Overshot a bit; go back the other way.** At `weight = 0.88` the error is 0.03, so the weight turns back.
3. **Overshot again; go back, but only a little.** At `weight = 0.69` the error is 0.002.
4. **Pretty much there.** At `weight = 0.73` the error is about 0.000009.

> Several steps of learning, part 1 (steps 1 and 2)
>
> <img src="pictures/chapter 4/picture 4.8.png" alt="Figure 4.8" width="600"/>

> Several steps of learning, part 2 (steps 3 and 4)
>
> <img src="pictures/chapter 4/picture 4.9.png" alt="Figure 4.9" width="600"/>


In [10]:
weight, goal_pred, input = (0.0, 0.8, 1.1)

for iteration in range(4):
    print("-----\nWeight:" + str(weight))
    pred = input * weight
    error = (pred - goal_pred) ** 2
    delta = pred - goal_pred
    weight_delta = delta * input
    weight = weight - weight_delta
    print("Error:" + str(error) + " Prediction:" + str(pred))
    print("Delta:" + str(delta) + " Weight Delta:" + str(weight_delta))

-----
Weight:0.0
Error:0.6400000000000001 Prediction:0.0
Delta:-0.8 Weight Delta:-0.8800000000000001
-----
Weight:0.8800000000000001
Error:0.02822400000000005 Prediction:0.9680000000000002
Delta:0.16800000000000015 Weight Delta:0.1848000000000002
-----
Weight:0.6951999999999999
Error:0.0012446784000000064 Prediction:0.76472
Delta:-0.03528000000000009 Weight Delta:-0.0388080000000001
-----
Weight:0.734008
Error:5.4890317439999896e-05 Prediction:0.8074088
Delta:0.007408799999999993 Weight Delta:0.008149679999999992


---
## **10. Why Does This Work? What Is `weight_delta`, Really?**

### **10.1 Functions and Their Moving Parts**

A function takes some numbers as input and gives another number as output. It therefore defines a relationship between the input number(s) and the output number(s). Learning a function is powerful because it converts some numbers (say, image pixels) into other numbers (say, the probability that the image contains a cat). Consider:

In [11]:
def my_function(x):
    return x * 2

Every function has **moving parts**: pieces that can be tweaked to change the output. In `my_function`, what controls the relationship between input and output is the `2`. Now ask the same question about the error function:

```python
error = ((input * weight) - goal_pred) ** 2
```

Many things control the relationship between `input` and `error`: `goal_pred`, `input`, `**2`, `weight`, and all the parentheses and operations. Tweaking any of them would change the error. Which of them may legitimately be changed?

- **Changing `goal_pred`** to reduce the error is possible but silly: it is like "giving up" by setting goals to match capability. It denies that the network missed.
- **Changing `input`** until the error goes to 0 is akin to seeing the world as you want to see it instead of as it actually is. It changes the data until the network predicts what you want it to predict.
- **Changing the `2`, or the additions, subtractions, and multiplications** changes how the error is calculated in the first place. The error calculation would be meaningless if it no longer gave a good measure of how much was missed.

What is left is `weight`. Adjusting it does not change the perception of the world, the goal, or the error measure. Changing `weight` means the function **conforms to the patterns in the data**. With the rest of the function unchanged, the network is forced to model a pattern in the data. It may only change how the network *predicts*.

> **Key takeaway:** you can modify *anything* in the `pred` calculation except `input`.

The rest of the book, and the careers of many deep learning researchers, is about trying everything imaginable on the `pred` calculation so that it makes good predictions. Learning is all about automatically changing the prediction function so that the subsequent `error` goes down to 0.

### **10.2 Tunnel Vision on One Concept**

**Concept: learning is adjusting the weight to reduce the error to 0.** Knowing how to do this is all about understanding the *relationship* between `weight` and `error`, that is, how changing one variable changes the other. Another name for this relationship is **sensitivity**: the direction and amount that `error` changes when `weight` changes.

Hot and cold learning studies that relationship experimentally. It is like walking into a room with 15 unlabeled light switches and flipping them on and off to learn what each does. The network wiggled `weight` up and down, watched how `error` changed, and then moved `weight` using two `if` statements:

```python
if(down_error < up_error):
    weight = weight - step_amount

if(down_error > up_error):
    weight = weight + step_amount
```

But the combined `pred` and `error` formula already defines this relationship exactly. It is a formula; it is exact, computable, and universal:

```python
error = ((input * weight) - goal_pred) ** 2
```

The remaining question is how to use this formula to change `weight` so that `error` moves in a particular direction. As it turns out, a method exists for doing this for *any* formula.

---
## **11. Derivatives and How to Use Them to Learn**

### **11.1 A Box with Rods Poking Out of It**

Picture a cardboard box with two circular rods sticking out of two little holes. The blue rod sticks out by 2 inches and the red rod by 4 inches. The rods are connected, but in an unknown way. Pushing the blue rod in by 1 inch makes the red rod move in by 2 inches; pulling the blue rod back out by 1 inch pulls the red rod out by 2 inches. The relationship appears to be:

```python
red_length = blue_length * 2
```

The formal name for "when I tug on this part, how much does that other part move?" is a **derivative**. Here the derivative of red with respect to blue is **2**, the *multiplicative* relationship determined by the formula.

Important properties:

- A derivative is always **between two variables**: how one variable moves when the other is changed.
- If the derivative is **positive**, the two variables move in the *same* direction.
- If the derivative is **negative**, they move in *opposite* directions. (A derivative of -1 would make red move by the same amount as blue, but in the opposite direction.)

Given a function, the derivative therefore represents the direction and the amount that one variable changes if the other is changed. This is exactly what is needed.

### **11.2 Derivatives: Take Two**

A second way to explain derivatives is that a derivative is the **slope** at a point on a line or curve. If a function is plotted, the slope of the plotted line is the same thing as "how much one variable changes when the other changes". Plotting the error function with `input` and `goal_pred` fixed:

```python
error = ((0.5 * weight) - 0.8) ** 2
```

gives a big U-shaped curve. The point where `error == 0` is in the middle. To the right of that point the slope is positive, and to the left the slope is negative. The farther the `weight` is from the goal weight, the **steeper** the slope. These are useful properties:

- The **sign** of the slope gives the *direction*.
- The **steepness** of the slope gives the *amount*.

Both can be used to find the goal weight. The curve is what hot and cold learning would produce if every possible `weight` were tried and plotted. What is remarkable about derivatives is that they can look past the big formula for `error` and compute the slope (derivative) at any value of `weight`, which tells which direction reduces the error and gives some idea of how far away the optimal point (slope of zero) is.

> The error curve with starting weight and goal weight
>
> <img src="pictures/chapter 4/picture 4.10.png" alt="Figure 4.10" width="600"/>

### **11.3 What You Really Need to Know (and What You Do Not)**

With derivatives, any two variables in any formula can be picked and their interaction understood. Even for a "whopper" such as

```python
y = (((beta * gamma) ** 2) + (epsilon + 22 - x)) ** (1/2)
```

any two variables can be plotted against each other, and how one changes when the other changes can be computed. For any function, one can learn how to change one variable to move another in a particular direction.

This matters because **a neural network is really just a bunch of weights used to compute an error function**. For any error function, no matter how complicated, the relationship between any `weight` and the final `error` can be computed, and each `weight` can be changed to reduce `error` toward 0.

**What you do not really need to know: calculus.** Learning all the methods for taking any two variables in any function and computing their relationship takes about three semesters of college, and only a very small subset is needed for deep learning. Calculus is largely about memorizing derivative rules for every possible function. In this book, derivatives are looked up in a reference table, as is done in practice. What matters is what the derivative **represents**: the sensitivity between two variables. This can be positive (variables move together), negative (they move in opposite directions), or zero (one stays fixed regardless of the other, e.g. `y = 0 * x`).

### **11.4 How to Use a Derivative to Learn**

**`weight_delta` is your derivative.**

`error` is a measure of how much was missed. The derivative of `error` with respect to `weight` defines the relationship between each weight and how much was missed. In other words, it tells how much a weight contributed to the error.

On the error curve, the black dot is where `weight` starts (0.5), and the dotted circle is the goal weight. The dotted line attached to the black dot is the **slope**, the derivative, which tells how much `error` changes if `weight` changes at that point. It points downward, so it is a **negative slope**.

The slope of a line or curve always points in the direction opposite to the lowest point of the curve. So with a negative slope, `weight` should be *increased* to find the minimum of `error`. To find the minimum:

1. Compute the derivative of `error` with respect to `weight`.
2. Change `weight` in the **opposite direction** of that slope.

That moves the weight toward the minimum. The neural network learns. This method of finding error minimums is called **gradient descent**. The name is intuitive: `weight` is moved opposite the gradient value, increasing the weight when the gradient is negative and vice versa, **like gravity**.

> Using the derivative (`weight_delta`) to learn
>
> <img src="pictures/chapter 4/picture 4.11.png" alt="Figure 4.11" width="600"/>

The earlier "Look familiar?" listing is exactly this code, now identified as gradient descent: `weight_delta = delta * input` is the **derivative** (how fast the error changes given changes in the weight). The step-by-step illustrations (a big weight increase, then overshoot and return) are the same as in Figure 4.8.

---
## **12. Breaking Gradient Descent**

**Just give me the code!** Here is the working version again (starting at `weight = 0.5`, `input = 0.5`, `goal_pred = 0.8`). The book's output begins with `Error:0.3025 Prediction:0.25` and ends near `Prediction:0.7977`.

In [12]:
weight = 0.5
goal_pred = 0.8
input = 0.5

for iteration in range(20):
    pred = input * weight
    error = (pred - goal_pred) ** 2
    delta = pred - goal_pred
    weight_delta = input * delta
    weight = weight - weight_delta
    print("Error:" + str(error) + " Prediction:" + str(pred))

Error:0.30250000000000005 Prediction:0.25
Error:0.17015625000000004 Prediction:0.3875
Error:0.095712890625 Prediction:0.49062500000000003
Error:0.05383850097656251 Prediction:0.56796875
Error:0.03028415679931642 Prediction:0.6259765625
Error:0.0170348381996155 Prediction:0.669482421875
Error:0.00958209648728372 Prediction:0.70211181640625
Error:0.005389929274097089 Prediction:0.7265838623046875
Error:0.0030318352166796153 Prediction:0.7449378967285156
Error:0.0017054073093822882 Prediction:0.7587034225463867
Error:0.0009592916115275371 Prediction:0.76902756690979
Error:0.0005396015314842384 Prediction:0.7767706751823426
Error:0.000303525861459885 Prediction:0.7825780063867569
Error:0.00017073329707118678 Prediction:0.7869335047900676
Error:9.603747960254256e-05 Prediction:0.7902001285925507
Error:5.402108227642978e-05 Prediction:0.7926500964444131
Error:3.038685878049206e-05 Prediction:0.7944875723333098
Error:1.7092608064027242e-05 Prediction:0.7958656792499823
Error:9.614592036015323

Now that it works, the author encourages experimenting with the starting `weight`, `goal_pred`, and `input`: the network will figure out how to predict the output from the input using the weight. Trying to break something is a great way to learn about it.

Set `input = 2` and keep `goal_pred = 0.8`. The book's output starts with `Error:0.04 Prediction:1.0`, `Error:0.36 Prediction:0.2`, `Error:3.24 Prediction:2.6`, and ends with enormous values such as `Prediction:-232452292.6`.

In [13]:
weight = 0.5
goal_pred = 0.8
input = 2

for iteration in range(20):
    pred = input * weight
    error = (pred - goal_pred) ** 2
    delta = pred - goal_pred
    weight_delta = input * delta
    weight = weight - weight_delta
    print("Error:" + str(error) + " Prediction:" + str(pred))

Error:0.03999999999999998 Prediction:1.0
Error:0.3599999999999998 Prediction:0.20000000000000018
Error:3.2399999999999984 Prediction:2.5999999999999996
Error:29.159999999999986 Prediction:-4.599999999999999
Error:262.4399999999999 Prediction:16.999999999999996
Error:2361.959999999998 Prediction:-47.79999999999998
Error:21257.639999999978 Prediction:146.59999999999994
Error:191318.75999999983 Prediction:-436.5999999999998
Error:1721868.839999999 Prediction:1312.9999999999995
Error:15496819.559999991 Prediction:-3935.799999999999
Error:139471376.03999993 Prediction:11810.599999999997
Error:1255242384.3599997 Prediction:-35428.59999999999
Error:11297181459.239996 Prediction:106288.99999999999
Error:101674633133.15994 Prediction:-318863.79999999993
Error:915071698198.4395 Prediction:956594.5999999997
Error:8235645283785.954 Prediction:-2869780.599999999
Error:74120807554073.56 Prediction:8609344.999999996
Error:667087267986662.1 Prediction:-25828031.799999986
Error:6003785411879960.0 Predi

The predictions **exploded**. They alternate between negative and positive, getting farther from the true answer at every step. In other words, every update to the weight overcorrects.

### **12.1 Visualizing the Overcorrections**

With `input = 2.0`, the weight starts at 0.5 with an error of 0.04. The first update overshoots to `weight = 0.1` (error 0.36), and the next overshoots to `weight = 1.3` (error 3.24). Each step lands farther from the bottom of the bowl than the one before it.

> Overcorrections when input = 2.0
>
> <img src="pictures/chapter 4/picture 4.12.png" alt="Figure 4.12" width="600"/>

### **12.2 Divergence**

**Sometimes neural networks explode in value.**

> Divergence: the derivative grows with each step
>
> <img src="pictures/chapter 4/picture 4.13.png" alt="Figure 4.13" width="600"/>

The explosion in the error was caused by making the input larger. Consider how the weight is updated:

```python
weight = weight - (input * (pred - goal_pred))
```

If the `input` is large enough, the weight update can be large **even when the error is small**. A large weight update combined with a small error makes the network overcorrect. If the new error is even bigger, it overcorrects even more. This phenomenon is called **divergence**.

With a big input, the prediction is very sensitive to changes in the weight (because `pred = input * weight`). Even though the weight is still starting at 0.5, the derivative at that point is very steep, which is why the U-shaped error curve is so tight. The error is very sensitive to the weight: the derivative is really big. How can it be made smaller?

---
## **13. Divergence and Alpha**

### **13.1 Introducing Alpha**

**It is the simplest way to prevent overcorrecting weight updates.**

The problem: if the input is too big, the weight update can overcorrect. The symptom: when it overcorrects, the new derivative is even larger in magnitude than when the process started (although the sign is opposite). Looking back at the divergence graph, step 2 is farther from the goal than step 1, so the derivative is even greater, which pushes step 3 even farther from the goal than step 2. The network continues like this, demonstrating divergence.

The symptom is the overshooting, and the solution is to **multiply the weight update by a fraction** to make it smaller. In most cases this means multiplying the update by a single real number between 0 and 1, known as **alpha**. This has no effect on the core issue (that the input is large), and it also reduces weight updates for inputs that are not too large.

Finding the appropriate alpha, even for state-of-the-art networks, is often done by **guessing**:

- If the error starts **diverging** (going up), alpha is too high and should be decreased.
- If learning is happening **too slowly**, alpha is too low and should be increased.

Other methods attempt to counter this issue, but plain gradient descent is still very popular.

### **13.2 Alpha in Code**

The weights were being updated according to:

```python
weight = weight - derivative
```

Accounting for alpha is a small change. If alpha is small (say, 0.01), it reduces the weight update considerably and thereby prevents overshooting:

```python
weight = weight - (alpha * derivative)
```

Installing alpha into the tiny implementation from the beginning of the chapter and running it with `input = 2` (which did not work before) gives the cell below. The book invites experimentation: what happens when alpha is made crazy small or big, or even negative? The output begins with `Error:0.04 Prediction:1.0`, `Error:0.0144 Prediction:0.92`, and ends near `Prediction:0.800012187195`.

In [14]:
weight = 0.5
goal_pred = 0.8
input = 2
alpha = 0.1

for iteration in range(20):
    pred = input * weight
    error = (pred - goal_pred) ** 2
    derivative = input * (pred - goal_pred)
    weight = weight - (alpha * derivative)

    print("Error:" + str(error) + " Prediction:" + str(pred))

Error:0.03999999999999998 Prediction:1.0
Error:0.0144 Prediction:0.92
Error:0.005183999999999993 Prediction:0.872
Error:0.0018662400000000014 Prediction:0.8432000000000001
Error:0.0006718464000000028 Prediction:0.8259200000000001
Error:0.00024186470400000033 Prediction:0.815552
Error:8.70712934399997e-05 Prediction:0.8093312
Error:3.134566563839939e-05 Prediction:0.80559872
Error:1.1284439629823931e-05 Prediction:0.803359232
Error:4.062398266736526e-06 Prediction:0.8020155392
Error:1.4624633760252567e-06 Prediction:0.8012093235200001
Error:5.264868153690924e-07 Prediction:0.8007255941120001
Error:1.8953525353291194e-07 Prediction:0.8004353564672001
Error:6.82326912718715e-08 Prediction:0.8002612138803201
Error:2.456376885786678e-08 Prediction:0.8001567283281921
Error:8.842956788836216e-09 Prediction:0.8000940369969153
Error:3.1834644439835434e-09 Prediction:0.8000564221981492
Error:1.1460471998340758e-09 Prediction:0.8000338533188895
Error:4.125769919393652e-10 Prediction:0.80002031199

The tiniest neural network can once again make good predictions. How was alpha = 0.1 chosen? The author simply tried it and it worked. Despite the many advances in deep learning, most people try several orders of magnitude of alpha (10, 1, 0.1, 0.01, 0.001, 0.0001) and then tweak from the best result. It is more art than science for now, and there are more advanced methods that come later.

---
## **14. Memorizing**

**It's time to really learn this stuff.**

The author strongly recommends an exercise: try to build the code of the previous section in a Jupyter notebook (or a `.py` file) **from memory**. It may seem like overkill, but the author did not have the "click" moment with neural networks until being able to do this.

There are two reasons it matters:

- The only way to know that all the information in the chapter has been absorbed is to produce it from your own head. Neural networks have many small moving parts, and it is easy to miss one.
- Later chapters refer to the concepts of this chapter at a faster pace, so that more time can be spent on newer material. When the book says "add your alpha parameterization to the weight update", the reader should immediately recognize which concepts are being referred to.

Memorizing small bits of neural network code has been hugely beneficial for the author and for many others who took this advice.

---
## **15. Summary**

*(The book's Chapter 4 does not end with a formal summary section; this one condenses the chapter.)*

Chapter 4 explains how a network learns by completing the *predict, compare, learn* cycle. The key takeaways are:

- **Compare** measures how much a prediction missed by. This chapter uses **mean squared error**: `error = (pred - goal_pred) ** 2`. Squaring keeps the error positive (so errors do not cancel when averaged) and amplifies big errors while reducing small ones.
- **Learn** is **error attribution**: working out how each weight contributed to the error, and then moving each weight accordingly.
- **Hot and cold learning** wiggles the weight up and down, keeps whichever direction reduces the error, and repeats. It is simple but **inefficient** (three predictions per update) and the fixed `step_amount` has nothing to do with the error, so it can overshoot and never reach the exact goal.
- Learning is a **search problem**: searching for the weight configuration that brings the error to 0.
- Combining the prediction and error formulas defines an **exact relationship** between `error` and `weight` (a bowl-shaped curve). Its **derivative**, the slope, gives both the **direction** (sign) and the **amount** (steepness) in which to move the weight.
- **Gradient descent** moves the weight opposite to the derivative: `delta = pred - goal_pred`, `weight_delta = delta * input`, `weight -= weight_delta`. Multiplying by `input` provides **scaling, negative reversal, and stopping**.
- Only the `weight` (the prediction calculation, except `input`) may be modified; `goal_pred`, `input`, and the error logic must stay fixed so that the network conforms to the patterns in the data.
- With large inputs, the update can overcorrect and the network can **diverge**. The fix is **alpha**, a small number between 0 and 1: `weight = weight - (alpha * derivative)`. Alpha is usually found by trying several orders of magnitude and tuning from there.

Deep calculus is not required: it is enough to understand that a derivative is the sensitivity between two variables. The author recommends memorizing the small code segments of this chapter, because later chapters build on them at a faster pace.